<a href="https://colab.research.google.com/github/Andrew-MSU/web-gis-automations/blob/main/GPT_lecture_for_me.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install dependencies (unchanged)
!pip install python-pptx openai gtts moviepy pdf2image
!apt-get update
!apt-get install -y poppler-utils ffmpeg
!which pdfinfo
!pdfinfo -v

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [2]:
# Cell 2: Import libraries and set up environment
import os
import base64
from pptx import Presentation
from openai import OpenAI
from gtts import gTTS
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips
from google.colab import files
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

BASE_DIR = "/content"
INPUT_DIR = os.path.join(BASE_DIR, "input")
IMAGES_DIR = os.path.join(BASE_DIR, "images")
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")
AUDIOS_DIR = os.path.join(BASE_DIR, "audios")
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")

for directory in [INPUT_DIR, IMAGES_DIR, SCRIPTS_DIR, AUDIOS_DIR, VIDEOS_DIR]:
    if not os.path.exists(directory):
        os.makedirs(directory)

def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image {image_path}: {str(e)}")
        return None

### Instructions for Users
Open your PowerPoint file in Microsoft PowerPoint.
Export it as a PDF (File > Export > PDF). Ensure you export "All Slides" and use the standard layout.
Upload both the .pptx file and the .pdf file to Colab when prompted.

In [3]:
# Cell 3: Upload files and generate images from PDF, with resizing
print("Upload your PowerPoint file (e.g., sample.pptx):")
uploaded_ppt = files.upload()
ppt_filename = list(uploaded_ppt.keys())[0]
ppt_path = os.path.join(INPUT_DIR, ppt_filename)
with open(ppt_path, "wb") as f:
    f.write(uploaded_ppt[ppt_filename])

print("Upload the PDF version of your PowerPoint (e.g., sample.pdf):")
uploaded_pdf = files.upload()
pdf_filename = list(uploaded_pdf.keys())[0]
pdf_path = os.path.join(INPUT_DIR, pdf_filename)
with open(pdf_path, "wb") as f:
    f.write(uploaded_pdf[pdf_filename])

try:
    print("Converting PDF pages to images...")
    images = convert_from_path(pdf_path, dpi=200)
    if not images:
        raise ValueError("No images generated from PDF. Is the PDF empty or corrupted?")

    # Resize images to a compatible resolution (e.g., 1920x1080)
    target_width = 1920
    target_height = 1080
    from PIL import Image
    for i, image in enumerate(images):
        # Convert to RGB if not already (pdf2image sometimes outputs RGBA)
        image = image.convert("RGB")

        # Resize while maintaining aspect ratio
        image.thumbnail((target_width, target_height), Image.Resampling.LANCZOS)

        # Create a new blank canvas with target dimensions (1920x1080) and paste the resized image centered
        new_image = Image.new("RGB", (target_width, target_height), (255, 255, 255))  # White background
        paste_xy = ((target_width - image.width) // 2, (target_height - image.height) // 2)
        new_image.paste(image, paste_xy)

        # Save the resized image
        image_path = os.path.join(IMAGES_DIR, f"slide_{i + 1}.png")
        new_image.save(image_path, "PNG")
        print(f"Saved slide image: {image_path} (resized to {new_image.width}x{new_image.height})")
except Exception as e:
    print(f"Error converting PDF to images: {str(e)}")
    raise

Upload your PowerPoint file (e.g., sample.pptx):


Saving 1_Foundations.pptx to 1_Foundations (1).pptx
Upload the PDF version of your PowerPoint (e.g., sample.pdf):


Saving 1_Foundations.pdf to 1_Foundations (1).pdf
Converting PDF pages to images...
Saved slide image: /content/images/slide_1.png (resized to 1920x1080)
Saved slide image: /content/images/slide_2.png (resized to 1920x1080)
Saved slide image: /content/images/slide_3.png (resized to 1920x1080)
Saved slide image: /content/images/slide_4.png (resized to 1920x1080)
Saved slide image: /content/images/slide_5.png (resized to 1920x1080)
Saved slide image: /content/images/slide_6.png (resized to 1920x1080)
Saved slide image: /content/images/slide_7.png (resized to 1920x1080)
Saved slide image: /content/images/slide_8.png (resized to 1920x1080)
Saved slide image: /content/images/slide_9.png (resized to 1920x1080)
Saved slide image: /content/images/slide_10.png (resized to 1920x1080)
Saved slide image: /content/images/slide_11.png (resized to 1920x1080)
Saved slide image: /content/images/slide_12.png (resized to 1920x1080)


In [4]:
# Cell 4: Parse PowerPoint (unchanged)
def parse_ppt(ppt_path):
    prs = Presentation(ppt_path)
    slide_data = []
    num_images = len([f for f in os.listdir(IMAGES_DIR) if f.startswith("slide_")])
    if num_images != len(prs.slides):
        print(f"Warning: Number of slides ({len(prs.slides)}) does not match number of PDF pages ({num_images})")
    for slide_num, slide in enumerate(prs.slides):
        slide_text = ""
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                slide_text += shape.text + "\n"
        notes_text = ""
        if slide.notes_slide.notes_text_frame:
            notes_text = slide.notes_slide.notes_text_frame.text
        slide_image_path = os.path.join(IMAGES_DIR, f"slide_{slide_num + 1}.png")
        if not os.path.exists(slide_image_path):
            print(f"Warning: Image for slide {slide_num + 1} not found at {slide_image_path}")
            continue
        slide_data.append({
            "slide_num": slide_num + 1,
            "text": slide_text,
            "notes": notes_text,
            "image_path": slide_image_path
        })
    return slide_data

slides = parse_ppt(ppt_path)

In [5]:
# Cell 5: Generate initial scripts with OpenAI gpt-4o, including images
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def generate_initial_script(slide_text, notes_text, image_path, slide_num):
    client = OpenAI(api_key=OPENAI_API_KEY)
    text_prompt = (
        "You are a lecturer preparing a script for a slide in a presentation. "
        "The slide content is:\n\n"
        f"Slide Text: {slide_text}\n"
        f"Speaker Notes: {notes_text}\n\n"
        "Below is the slide image. Please generate a clear and concise script that explains the slide in a way suitable for a lecture, "
        "covering key points in a first-person tone. Include references to the image content where relevant (e.g., describe charts, diagrams, or visuals). "
        "Keep it under 200 words."
        "Think about what concepts an undergraduate student might need a more detailed explanation for, and provide it"
    )
    base64_image = encode_image_to_base64(image_path) if image_path and os.path.exists(image_path) else None
    messages = [
        {"role": "system", "content": "You are a helpful lecturer."},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text_prompt}
            ]
        }
    ]
    if base64_image:
        messages[1]["content"].append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{base64_image}"
            }
        })
    else:
        print(f"Warning: No image available for Slide {slide_num}. Generating script without image input.")
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            max_tokens=150
        )
        script = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating initial script for Slide {slide_num}: {str(e)}")
        script = "Failed to generate initial script due to an error."
    initial_script_path = os.path.join(SCRIPTS_DIR, f"initial_script_slide_{slide_num}.txt")
    with open(initial_script_path, "w", encoding="utf-8") as f:
        f.write(script)
    return script, initial_script_path

initial_scripts = []
for slide in slides:
    print(f"Generating initial script for Slide {slide['slide_num']}...")
    script, script_path = generate_initial_script(
        slide["text"],
        slide["notes"],
        slide["image_path"],
        slide["slide_num"]
    )
    print(f"Initial script saved to: {script_path}")
    print(f"Initial script: {script}\n")
    initial_scripts.append({
        "slide_num": slide["slide_num"],
        "text": slide["text"],
        "notes": slide["notes"],
        "image_path": slide["image_path"],
        "initial_script": script,
        "initial_script_path": script_path
    })

Generating initial script for Slide 1...
Initial script saved to: /content/scripts/initial_script_slide_1.txt
Initial script: Welcome to GEO 315, focusing on geological structures. I’m Dr. Andrew Laskowski, or Drew, your Associate Professor in Earth Sciences. This course delves into the foundations of structural geology and tectonics, exploring how the Earth's crust deforms and the processes behind it. 

The image shown on the slide highlights key contents from our primary resource, “Processes in Structural Geology and Tectonics” by van der Pluijm and Marshak. This text is crucial for understanding topics like geological structures, frictional and plastic regimes, and folding processes. 

We’ll cover essential geological concepts and their practical implications. Familiarity with terms like faulting, folding, and deformation regimes will be integral. The site featured in the image

Generating initial script for Slide 2...
Initial script saved to: /content/scripts/initial_script_slide_2

In [6]:
# Cell 5.5: Refine scripts for coherence, including images with OpenAI gpt-4o
def refine_scripts_with_coherence(initial_scripts):
    client = OpenAI(api_key=OPENAI_API_KEY)
    full_context = "Below is the draft of a presentation with scripts for each slide:\n\n"
    for item in initial_scripts:
        full_context += (
            f"Slide {item['slide_num']}:\n"
            f"Slide Text: {item['text']}\n"
            f"Speaker Notes: {item['notes']}\n"
            f"Initial Script: {item['initial_script']}\n\n"
        )
    refined_scripts = []
    for idx, item in enumerate(initial_scripts):
        slide_num = item['slide_num']
        print(f"Refining script for Slide {slide_num}...")
        prompt = (
            f"I am a lecturer preparing a script for a coherent presentation. "
            "Below is the full draft of my presentation with initial scripts for each slide:\n\n"
            f"{full_context}\n\n"
            f"Now, refine the script for Slide {slide_num} to ensure it flows naturally within the presentation. "
            "Consider what has been discussed in previous slides and preview what is upcoming in future slides. "
            "Take into account the slide images when refining the script, referencing visuals where relevant. "
            "Add transitions if necessary to connect with prior and upcoming content. "
            "Keep the tone professional, first-person, and concise (under 200 words). "
            "Focus on the content of Slide {slide_num}, but make it part of a cohesive lecture."
            "Suppress any response related to the prompt, just provide the refined script."
        )
        base64_image = encode_image_to_base64(item['image_path']) if item['image_path'] and os.path.exists(item['image_path']) else None
        messages = [
            {"role": "system", "content": "You are a helpful lecturer ensuring a cohesive presentation."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        if base64_image:
            messages[1]["content"].append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_image}"
                }
            })
        else:
            print(f"Warning: No image available for Slide {slide_num} during refinement.")
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                max_tokens=150
            )
            refined_script = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error refining script for Slide {slide_num}: {str(e)}")
            refined_script = item['initial_script']
        script_path = os.path.join(SCRIPTS_DIR, f"script_slide_{slide_num}.txt")
        with open(script_path, "w", encoding="utf-8") as f:
            f.write(refined_script)
        print(f"Refined script saved to: {script_path}")
        print(f"Refined script: {refined_script}\n")
        refined_scripts.append({
            "slide_num": slide_num,
            "text": item["text"],
            "notes": item["notes"],
            "image_path": item["image_path"],
            "refined_script": refined_script,
            "script_path": script_path
        })
    return refined_scripts

refined_scripts = refine_scripts_with_coherence(initial_scripts)

Refining script for Slide 1...
Refined script saved to: /content/scripts/script_slide_1.txt
Refined script: Welcome to GEO 315, where we delve into geological structures. I’m Dr. Andrew Laskowski, or Drew, your Associate Professor in Earth Sciences. Today, we begin our exploration of structural geology and tectonics, focusing on the deformation of the Earth's crust and underlying processes. 

Our primary resource is “Processes in Structural Geology and Tectonics” by van der Pluijm and Marshak, seen here. This text will guide us through key concepts like faulting, folding, and deformation regimes—subjects essential for understanding the dynamics of our planet. As we progress, we'll discuss these elements in detail, connecting them with real-world implications and applications. 

From here, we will transition to discuss specific geological questions and exercises designed

Refining script for Slide 2...
Refined script saved to: /content/scripts/script_slide_2.txt
Refined script: As we tr

In [7]:
# Cell 6: Generate audio with gTTS
def generate_audio(script_path, slide_num):
    """Generate audio from a script using gTTS."""
    with open(script_path, "r", encoding="utf-8") as f:
        script = f.read()

    try:
        tts = gTTS(text=script, lang="en")
        audio_path = os.path.join(AUDIOS_DIR, f"audio_slide_{slide_num}.mp3")
        tts.save(audio_path)
        return audio_path
    except Exception as e:
        print(f"Error generating audio for Slide {slide_num}: {str(e)}")
        # Fallback: Return a placeholder path (or handle error as needed)
        audio_path = os.path.join(AUDIOS_DIR, f"audio_slide_{slide_num}_error.mp3")
        with open(audio_path, "w") as f:
            f.write("")
        return audio_path

# Generate audio for each script
for slide in slides:
    script_path = os.path.join(SCRIPTS_DIR, f"script_slide_{slide['slide_num']}.txt")
    print(f"Generating audio for Slide {slide['slide_num']}...")
    audio_path = generate_audio(script_path, slide["slide_num"])
    print(f"Audio saved to: {audio_path}\n")

Generating audio for Slide 1...
Audio saved to: /content/audios/audio_slide_1.mp3

Generating audio for Slide 2...
Audio saved to: /content/audios/audio_slide_2.mp3

Generating audio for Slide 3...
Audio saved to: /content/audios/audio_slide_3.mp3

Generating audio for Slide 4...
Audio saved to: /content/audios/audio_slide_4.mp3

Generating audio for Slide 5...
Audio saved to: /content/audios/audio_slide_5.mp3

Generating audio for Slide 6...
Audio saved to: /content/audios/audio_slide_6.mp3

Generating audio for Slide 7...
Audio saved to: /content/audios/audio_slide_7.mp3

Generating audio for Slide 8...
Audio saved to: /content/audios/audio_slide_8.mp3

Generating audio for Slide 9...
Audio saved to: /content/audios/audio_slide_9.mp3

Generating audio for Slide 10...
Audio saved to: /content/audios/audio_slide_10.mp3

Generating audio for Slide 11...
Audio saved to: /content/audios/audio_slide_11.mp3

Generating audio for Slide 12...
Audio saved to: /content/audios/audio_slide_12.mp3

In [ ]:
# Cell 7: Assemble video
def assemble_video(slides):
    clips = []
    for slide in slides:
        slide_num = slide["slide_num"]
        image_path = slide["image_path"]
        audio_path = os.path.join(AUDIOS_DIR, f"audio_slide_{slide_num}.mp3")
        if not os.path.exists(image_path):
            print(f"Error: Image for slide {slide_num} not found. Skipping...")
            continue
        audio = AudioFileClip(audio_path)
        duration = audio.duration
        image_clip = ImageClip(image_path, duration=duration)
        image_clip = image_clip.set_audio(audio)
        clips.append(image_clip)
    if not clips:
        raise ValueError("No valid clips to concatenate.")
    final_clip = concatenate_videoclips(clips, method="compose")
    output_path = os.path.join(VIDEOS_DIR, "lecture_video.mp4")
    final_clip.write_videofile(
        output_path,
        fps=24,
        codec="libx264",
        audio_codec="aac",
        ffmpeg_params=["-pix_fmt", "yuv420p", "-profile:v", "main", "-level", "4.0"],
        preset="medium",
        bitrate="1000k",
        audio_bitrate="192k",
        verbose=False
    )
    return output_path

print("Assembling video...")
video_path = assemble_video(slides)
print(f"Final video saved to: {video_path}")

Assembling video...
Moviepy - Building video /content/videos/lecture_video.mp4.
MoviePy - Writing audio in lecture_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/videos/lecture_video.mp4



t:   2%|▏         | 335/17210 [00:31<22:37, 12.43it/s, now=None]

In [ ]:
# Cell 8: Download video (unchanged)
files.download(video_path)